In [2]:
import torch
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import gpytorch
import xarray as xr
import matplotlib.pyplot as plt

# Load ice velocity nc file

In [3]:
# Load dataset
vel = xr.open_dataset("antarctic_ice_vel_phase_map_v01.nc")

In [4]:
vel

<xarray.Dataset>
Dimensions:       (x: 12445, y: 12445)
Coordinates:
  * x             (x) float64 -2.8e+06 -2.8e+06 -2.799e+06 ... 2.799e+06 2.8e+06
  * y             (y) float64 2.8e+06 2.8e+06 2.799e+06 ... -2.799e+06 -2.8e+06
    lat           (y, x) float64 ...
    lon           (y, x) float64 ...
Data variables:
    coord_system  |S1 ...
    VX            (y, x) float32 ...
    VY            (y, x) float32 ...
    STDX          (y, x) float32 ...
    STDY          (y, x) float32 ...
    ERRX          (y, x) float32 ...
    ERRY          (y, x) float32 ...
    CNT           (y, x) int32 ...
    SOURCE        (y, x) int8 ...
Attributes: (12/27)
    Conventions:               CF-1.6
    Metadata_Conventions:      CF-1.6, Unidata Dataset Discovery v1.0, GDS v2.0
    standard_name_vocabulary:  CF Standard Name Table (v22, 12 February 2013)
    id:                        v_mix.v8Jul2019.nc
    title:                     MEaSURES Antarctica Ice Velocity Map 450m spacing
    product_version:            
    ...                        ...
    time_coverage_start:       1995-01-01
    time_coverage_end:         2016-12-31
    project:                   NASA/MEaSUREs
    creator_name:              J. Mouginot
    comment:                    
    license:                   No restrictions on access or use.

In [5]:
# Finetune these
transant_x_min = - 333000.0
transant_y_min = - 999500.0
transant_x_max = 366500.0
transant_y_max = 0.

# Top right corner: [366500, 0]
# Bottom left corner [- 333000, - 999500]

domec_x_min = 800000
domec_y_min = -1330000
domec_x_max = 1600000
domec_y_max = -600000

In [6]:
# y slicing ordering is (max, min)
vel.sel(x = slice(domec_x_min, domec_x_max), y = slice(domec_y_max, domec_y_min))

<xarray.Dataset>
Dimensions:       (x: 1778, y: 1622)
Coordinates:
  * x             (x) float64 8e+05 8.004e+05 8.009e+05 ... 1.599e+06 1.6e+06
  * y             (y) float64 -6.002e+05 -6.006e+05 ... -1.329e+06 -1.33e+06
    lat           (y, x) float64 ...
    lon           (y, x) float64 ...
Data variables:
    coord_system  |S1 ...
    VX            (y, x) float32 ...
    VY            (y, x) float32 ...
    STDX          (y, x) float32 ...
    STDY          (y, x) float32 ...
    ERRX          (y, x) float32 ...
    ERRY          (y, x) float32 ...
    CNT           (y, x) int32 ...
    SOURCE        (y, x) int8 ...
Attributes: (12/27)
    Conventions:               CF-1.6
    Metadata_Conventions:      CF-1.6, Unidata Dataset Discovery v1.0, GDS v2.0
    standard_name_vocabulary:  CF Standard Name Table (v22, 12 February 2013)
    id:                        v_mix.v8Jul2019.nc
    title:                     MEaSURES Antarctica Ice Velocity Map 450m spacing
    product_version:            
    ...                        ...
    time_coverage_start:       1995-01-01
    time_coverage_end:         2016-12-31
    project:                   NASA/MEaSUREs
    creator_name:              J. Mouginot
    comment:                    
    license:                   No restrictions on access or use.

In [8]:
vel_slice = vel.sel(x = slice(transant_x_min, transant_x_max), y = slice(transant_y_max, transant_y_min))
scene = vel_slice.isel(y = slice(0, 50), x = slice(0, 50))

scene_vel_tensor = torch.tensor(scene.vel.values)

<xarray.Dataset>
Dimensions:       (x: 50, y: 50)
Coordinates:
  * x             (x) float64 -3.326e+05 -3.322e+05 ... -3.11e+05 -3.106e+05
  * y             (y) float64 -350.0 -800.0 -1.25e+03 ... -2.195e+04 -2.24e+04
    lat           (y, x) float64 ...
    lon           (y, x) float64 ...
Data variables:
    coord_system  |S1 ...
    VX            (y, x) float32 ...
    VY            (y, x) float32 ...
    STDX          (y, x) float32 ...
    STDY          (y, x) float32 ...
    ERRX          (y, x) float32 ...
    ERRY          (y, x) float32 ...
    CNT           (y, x) int32 ...
    SOURCE        (y, x) int8 ...
Attributes: (12/27)
    Conventions:               CF-1.6
    Metadata_Conventions:      CF-1.6, Unidata Dataset Discovery v1.0, GDS v2.0
    standard_name_vocabulary:  CF Standard Name Table (v22, 12 February 2013)
    id:                        v_mix.v8Jul2019.nc
    title:                     MEaSURES Antarctica Ice Velocity Map 450m spacing
    product_version:            
    ...                        ...
    time_coverage_start:       1995-01-01
    time_coverage_end:         2016-12-31
    project:                   NASA/MEaSUREs
    creator_name:              J. Mouginot
    comment:                    
    license:                   No restrictions on access or use.

In [51]:
dims = 50

scene_vel_tensor = torch.cat((torch.tensor(scene.VX.values).unsqueeze(0), 
                              torch.tensor(scene.VY.values).unsqueeze(0),
                              torch.tensor(scene.coords["y"].values).unsqueeze(-1).repeat(1, dims).unsqueeze(0),
                              torch.tensor(scene.coords["x"].values).repeat(dims, 1).unsqueeze(0)),
                              dim = 0)

torch.save(scene_vel_tensor, './torch_data/scene_vel_tensor.pt')